In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 51. Week 35 — Self-attention, masks, and transformer boundaries

## 学習目標

- scaled dot-product attentionを行列で実装できる
- row-stochastic weightとcausal maskを監査できる
- positional informationが必要な理由を反例で示せる
- pretrained/fine-tuned modelをCoreのsmall attentionと区別できる

## 前提知識

- matrix calculus、softmax
- Week 34のsequence representation
- training-only vocabulary contract

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 51


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
fixture = qt.load_sec_teaching_fixture()
train_mask = fixture.training_mask
validation_mask = fixture.validation_mask

assert train_mask.sum() == 192
assert validation_mask.sum() == 64
assert not np.any(fixture.target_available_dates >= np.datetime64("2023-10-23"))
assert set(fixture.partitions) == {"inner_train", "inner_validation"}

print("fixture rows:", fixture.targets.size)
print("inner train / validation:", int(train_mask.sum()), int(validation_mask.sum()))
print("numeric / sequence shape:", fixture.numeric_features.shape, fixture.token_hashes.shape)
print("locked outer rows present: False")
print("fixture hash lineage:", fixture.provenance)

fixture rows: 256
inner train / validation: 192 64
numeric / sequence shape: (256, 12) (256, 128)
locked outer rows present: False
fixture hash lineage: {'panel_artifact_sha256': '6c6008c2f28c30299e15e37613cfb0b3b22e8fd283858f5b459227c7e4a412a8', 'previous_filing_sidecar_sha256': '9ff2efef335357ff53bb1e4ba5c57f4b2e8799fc4ee5d830c55843a50026fbbc', 'normalized_manifest_sha256': '1283b9cb0992cfd2caaa942f6c869e212762c90a9abbc9a050173f5e3963daba', 'preanalysis_contract_sha256': 'fbe69fdf3b3bccba7fab70bcbb726d0df61685901cc0322d76fc66be1d7bbd6e'}


## 1. Scaled dot-product attention

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V,
$$

$$
\operatorname{Attention}(X)=
\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.
$$

mask (M) は許可しない位置へ (-\infty) を置く。softmaxは各query rowで和1にならなければならない。SEC previous filingのdocument-level regressionでは必ずしもcausal maskが必要ではないが、autoregressive説明と混同しないため両方を実装する。

In [4]:
toy_embeddings = np.array(
    [
        [
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0],
            [1.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0],
        ]
    ]
)
identity = np.eye(4)
toy_output, toy_weights = qt.self_attention(
    toy_embeddings, identity, identity, identity, causal=True
)
assert np.allclose(toy_weights.sum(axis=-1), 1.0)
assert np.allclose(np.triu(toy_weights[0], k=1), 0.0)

fig = go.Figure(
    data=go.Heatmap(z=toy_weights[0], colorscale="Blues", zmin=0.0, zmax=1.0)
)
fig.update_layout(
    title="Causal attention audit",
    xaxis_title="Key position",
    yaxis_title="Query position",
    template="plotly_white",
)
fig.show()

## 2. Positionとsmall attention probe

self-attention単体はtoken permutationに対してequivariantである。位置を識別するにはpositional encodingを加える。ここでは幅8の固定embeddingにsin/cos positionを加え、single-head forward representationを平均poolしてridge probeへ渡す。これはpretrained transformerでも正式candidateでもない。

In [5]:
width = 8
embedded = qt.token_embedding(fixture.token_hashes, width, seed=20260811)
positions = np.arange(embedded.shape[1])[:, None]
frequencies = np.exp(-np.arange(0, width, 2) * np.log(10000.0) / width)
position_encoding = np.zeros((embedded.shape[1], width))
position_encoding[:, 0::2] = np.sin(positions * frequencies)
position_encoding[:, 1::2] = np.cos(positions * frequencies)
positioned = embedded + position_encoding[None, :, :]

attention_rng = task_rng(1)
query_weights = attention_rng.normal(scale=1.0 / np.sqrt(width), size=(width, width))
key_weights = attention_rng.normal(scale=1.0 / np.sqrt(width), size=(width, width))
value_weights = attention_rng.normal(scale=1.0 / np.sqrt(width), size=(width, width))
attention_output, attention_weights = qt.self_attention(
    positioned, query_weights, key_weights, value_weights, causal=False
)
attention_representation = attention_output.mean(axis=1)
attention_probe = qt.fit_sparse_ridge(
    attention_representation[train_mask], fixture.targets[train_mask], ridge=1.0
)
attention_prediction = attention_probe.predict(attention_representation[validation_mask])
attention_metrics = qt.regression_error_table(
    fixture.targets[validation_mask],
    attention_prediction,
    np.asarray(fixture.entity_ids)[validation_mask],
)
display(pd.DataFrame([{"model": "random_small_attention_probe", **attention_metrics}]))

mean_entropy = -np.mean(
    np.sum(attention_weights * np.log(np.maximum(attention_weights, 1e-15)), axis=-1)
)
print("mean attention entropy:", mean_entropy)
print("trainable parameters in this frozen probe: 0")

,model,mae,median_absolute_error,rmse,company_macro_mae
0,random_small_attention_probe,0.053271,0.025563,0.118545,0.047115


mean attention entropy: 4.7762707530318735
trainable parameters in this frozen probe: 0


## 3. Transformerとfoundation modelの境界

multi-head attentionはheadごとに異なるprojection subspaceを持つ。transformer blockはattentionだけでなくresidual connection、normalization、position-wise feed-forward networkを含む。pretraining、tokenizer、retrieval corpus、fine-tuningを省いたsingle-head NumPy実装を「foundation model」と呼ばない。

## 4. 失敗モード

- (1/\sqrt{d_k}) scalingを省きsoftmaxを飽和させる
- mask後にfuture weightがexact zeroか検査しない
- positionなしでtoken順序を学習したと主張する
- attention weightを因果的説明・feature importanceと呼ぶ
- pretrained modelのtraining corpus leakageを監査しない

## 5. 段階別演習

### 基礎

1. 各attention rowの和が1になる理由を説明せよ。
2. causal maskの上三角がzeroになるtestを書け。

### 標準

3. 2-head attentionを実装しparameter数を数えよ。
4. position encodingの有無でpermutation反例を作れ。

### 研究

5. pretrained encoder追加時のcorpus cutoff、license、carbon/compute budgetをmanifest化せよ。

## 6. Exit Criteria

- [ ] scaled dot-product attentionを実装した
- [ ] row sumとmaskをassertした
- [ ] positionの必要性を説明した
- [ ] attention weightを因果説明と呼んでいない
- [ ] small attentionとfoundation modelを区別した

## 7. 出典


- [Hochreiter and Schmidhuber (1997), Long Short-Term Memory](https://www.bioinf.jku.at/publications/older/2604.pdf)
- [Bai, Kolter, and Koltun (2018), An Empirical Evaluation of Generic Convolutional and Recurrent Networks](https://arxiv.org/abs/1803.01271)
- [Vaswani et al. (2017), Attention Is All You Need](https://arxiv.org/abs/1706.03762)